# Step 1. Preliminaries

## 1.1 Import Libraries

In [ ]:
# TODO

# 1.1 Import Libraries
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import linregress
from collections import defaultdict


# Deep Learning
import torch
import torch.nn as nn

# Custom Helper Functions
import helpers 

# Configuration
DATA_DIR = 'CamelinaMAGIC-Dataset-Complete'
EXPERIMENTS = [1, 2, 3] # We will process all three
RANDOMIZATION_FILE = os.path.join(DATA_DIR, 'CamelinaMAGIC-ALL-Randomization.xlsx')

print("Libraries imported and configuration set.")

## 1.2 Loading Data and Processing

In [ ]:
# 1.2 Data Loading and Processing


print("Loading data from Experiments:", EXPERIMENTS)

exp_transpiration_dataframes = {} # To store processed dataframes for each experiment

for exp_num in EXPERIMENTS:
    print(f"  Processing Experiment {exp_num}...")
    
    # 1. Define Paths
    transp_file = os.path.join(DATA_DIR, f'CamelinaMAGIC{exp_num}.0-DailyTranspiration.csv')
    weather_file = os.path.join(DATA_DIR, f'CamelinaMAGIC{exp_num}.0-WeatherStation.csv')

    try:
        df_transp = helpers.read_transpiration_csv(transp_file, RANDOMIZATION_FILE, exp_num)
        df_weather = helpers.read_weather_station_csv(weather_file, exp_num)

    except Exception as e:
        print(f"    Skipping Exp {exp_num} due to error: {e}")
        continue

    df_transp = df_transp[df_transp['Treatment'] == 'DR_100']     # Filter for DROUGHT Only Plants
    #helpers.plot_drought_transpiration_df(df_transp, title=f'Experiment {exp_num} - Before VPD Normalization')

    helpers.plot_drought_transpiration_df(df_transp, title=f'Experiment {exp_num} - Before Smoothing')
    #print(f"df_transp before smoothing for Exp {exp_num}:\n", df_transp.head())
    
    df_transp['Transp_original'] = df_transp['Transpiration']  # Keep original for comparison


    # Smooth Data (Rolling Average)

    # cols_to_smooth = ['Weight', 'Transpiration', 'Transp_Norm'] #old line
    cols_to_smooth = ['Transpiration']
    df_transp['Transpiration'] = df_transp.groupby('Sample')[cols_to_smooth].transform(   # Group by Sample to ensure we don't smooth across different plants
        lambda x: x.rolling(window=3, min_periods=1).mean()
    )
    helpers.plot_drought_transpiration_df(df_transp, title=f'Experiment {exp_num} - After Smoothing')
        
    #print(f"df_transp after smoothing for Exp {exp_num}:\n", df_transp.head())

    exp_transpiration_dataframes[exp_num] = df_transp # Store for later use.

print("Number of dataframes in the dict exp_transpiration_dataframes:", len(exp_transpiration_dataframes))



### Just to visualize important information for now

In [ ]:
combined_df = pd.concat([d for d in exp_transpiration_dataframes.values()], ignore_index=True)

unique_samples = combined_df['Sample'].unique()
print(f"  Found \033[32m\033[1m{len(unique_samples)}\033[0m\033[m unique samples in all the experiments combined:\n ", unique_samples, "\n")

unique_genotypes = combined_df['Genotype'].unique()
print(f"  Found \033[32m\033[1m{len(unique_genotypes)}\033[0m\033[m unique genotypes in all the experiments combined:\n ", unique_genotypes, "\n")


for exp, df in exp_transpiration_dataframes.items():
    unique_exp_samples = df['Sample'].unique()
    print(f"  Found \033[32m\033[1m{len(unique_exp_samples)}\033[0m\033[m unique samples in experiment \033[32m\033[1m{exp}\033[0m\033[m:\n ", unique_exp_samples, "\n")
    
    unique_exp_genotypes = df['Genotype'].unique()
    print(f"  Found \033[32m\033[1m{len(unique_exp_genotypes)}\033[0m\033[m unique genotypes in experiment \033[32m\033[1m{exp}\033[0m\033[m:\n ", unique_exp_genotypes, "\n")


### Setup correct date ranges for X input and Y target and build a dictionary to access experiment -> genotype -> timeseries

In [ ]:
unique_genotypes_per_exp = {
    exp: df['Genotype'].unique()
    for exp, df in exp_transpiration_dataframes.items()
}
print("Unique genotypes per experiment:\n", unique_genotypes_per_exp)

X_dict = {1:None, 2:None, 3:None}
y_dict = {1:None, 2:None, 3:None}

for e, df in exp_transpiration_dataframes.items():
    X_genexp = {}
    y_genexp = {}
    y_genexp_slope = {}
    for genotype in unique_genotypes_per_exp[e]:
        X_genexp[genotype] = exp_transpiration_dataframes[e].query('15 <= Days <= 25').query('Genotype == @genotype')
        y_genexp[genotype] = exp_transpiration_dataframes[e].query('36 <= Days <= 40').query('Genotype == @genotype')
        y_genexp_slope[genotype] = float(linregress(y_genexp[genotype]['Days'], y_genexp[genotype]['Transpiration']).slope)
    X_dict[e] = X_genexp
    y_dict[e] = y_genexp_slope

print(y_dict)

In [ ]:
# Train Linear Regression per genotype per experiment using LOOCV
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from scipy.stats import linregress
import numpy as np
import os
import pickle

# INPUT / RECOVERY windows (adjust if needed)
INPUT_WINDOW = list(range(15, 26))   # Days 15-25 inclusive (11 timesteps)
RECOVERY_WINDOW = list(range(36, 41)) # Days 36-40 inclusive

# Create models directory if it doesn't exist
MODELS_DIR = 'models'
if not os.path.exists(MODELS_DIR):
    os.makedirs(MODELS_DIR)
    print(f"Created '{MODELS_DIR}' directory for saving models.")

lr_results = []

for exp in [1, 2, 3]:
    gen_map = X_dict[exp]
    print(f"\n=== Experiment {exp} : {len(gen_map)} genotypes found ===")
    for genotype, df_X in gen_map.items():
        # df_X is expected to be a DataFrame with columns including 'Sample', 'Days', 'Transpiration' (or similar)
        if df_X is None or df_X.empty:
            print(f"Skipping Exp {exp} Genotype {genotype}: no data in X_dict")
            continue

        sample_features = []
        sample_targets = []
        sample_names = []

        # Group by sample to collect time-series per sample in the input window
        grouped = df_X.groupby('Sample')
        for sample, s_df in grouped:
            s_df_sorted = s_df.sort_values('Days')
            inp = s_df_sorted[s_df_sorted['Days'].isin(INPUT_WINDOW)]
            # require full coverage of INPUT_WINDOW - change min_timesteps if you want to be more permissive
            if len(inp) < len(INPUT_WINDOW):
                # skip incomplete samples
                continue
            # Use 'Transpiration' as feature; if you want multiple features, stack/concatenate here
            feat = inp['Transpiration'].values

            # Compute target slope for this sample from the recovery window using the full experiment dataframe
            exp_df = exp_transpiration_dataframes.get(exp)
            if exp_df is None:
                continue
            rec = exp_df[(exp_df['Sample'] == sample) & (exp_df['Days'].isin(RECOVERY_WINDOW))].sort_values('Days')
            if len(rec) < 2:
                continue
            slope = float(linregress(rec['Days'], rec['Transpiration']).slope)

            sample_names.append(sample)
            sample_features.append(feat)
            sample_targets.append(slope)

        n = len(sample_targets)
        if n < 2:
            print(f"Exp {exp} Genotype {genotype}: not enough samples after filtering ({n}), skipping.")
            continue

        X_arr = np.array(sample_features)  # shape (n_samples, seq_len)
        y_arr = np.array(sample_targets)

        # Flatten time-series for linear regression: (n, seq_len) -> (n, seq_len)
        # (already 2D) if you had multiple features per timestep you'd reshape to (n, seq_len * n_features)
        X_flat = X_arr.reshape(n, -1)

        # LOOCV
        loo = LeaveOneOut()
        preds = []
        actuals = []

        for train_idx, test_idx in loo.split(X_flat):
            X_train, X_test = X_flat[train_idx], X_flat[test_idx]
            y_train, y_test = y_arr[train_idx], y_arr[test_idx]

            scaler = StandardScaler()
            X_train_s = scaler.fit_transform(X_train)
            X_test_s = scaler.transform(X_test)

            model = LinearRegression()
            model.fit(X_train_s, y_train)
            y_pred = model.predict(X_test_s)[0]

            preds.append(y_pred)
            actuals.append(float(y_test[0]))

        # Metrics for this genotype-instance
        r2 = r2_score(actuals, preds)
        rmse = mean_squared_error(actuals, preds)
        mae = mean_absolute_error(actuals, preds)

        print(f"Exp {exp} | Genotype: {genotype} | samples={n} | R2={r2:.4f} | RMSE={rmse:.4f} | MAE={mae:.4f}")

        # Train final model on all data for saving
        scaler_final = StandardScaler()
        X_flat_scaled = scaler_final.fit_transform(X_flat)
        final_model = LinearRegression()
        final_model.fit(X_flat_scaled, y_arr)

        # Save model and scaler with the specified naming convention
        model_filename = f"LR_model_Cam_E{exp}_gen_{genotype.replace(' ', '_')}.pkl"
        scaler_filename = f"scaler_Cam_E{exp}_gen_{genotype.replace(' ', '_')}.pkl"
        model_path = os.path.join(MODELS_DIR, model_filename)
        scaler_path = os.path.join(MODELS_DIR, scaler_filename)

        with open(model_path, 'wb') as f:
            pickle.dump(final_model, f)
        with open(scaler_path, 'wb') as f:
            pickle.dump(scaler_final, f)

        print(f"  ✓ Saved model to: {model_path}")
        print(f"  ✓ Saved scaler to: {scaler_path}")

        lr_results.append({
            'Experiment': exp,
            'Genotype': genotype,
            'n_samples': n,
            'R2': r2,
            'RMSE': rmse,
            'MAE': mae,
            'sample_names': sample_names,
            'y_true': actuals,
            'y_pred': preds,
            'model_path': model_path,
            'scaler_path': scaler_path
        })

# Create a summary DataFrame
import pandas as pd
lr_results_df = pd.DataFrame([{
    'Experiment': r['Experiment'],
    'Genotype': r['Genotype'],
    'n_samples': r['n_samples'],
    'R2': r['R2'],
    'RMSE': r['RMSE'],
    'MAE': r['MAE'],
    'model_path': r['model_path']
} for r in lr_results])

if not lr_results_df.empty:
    display(lr_results_df.sort_values(['Experiment', 'R2'], ascending=[True, False]).reset_index(drop=True))
else:
    print('\nNo LR results were produced (no genotypes had enough samples).')

# Model Inference and Results Display

In [ ]:
# Model Inference: Load model and make predictions
import pickle
import os

# User Input
genotype = input("Enter genotype name: ").strip()
exp_num = int(input("Enter experiment number (1, 2, or 3): ").strip())
sample = input("Enter sample name (e.g., Camelina01): ").strip()

# Load the corresponding transpiration dataframe for this experiment
transp_file = os.path.join(DATA_DIR, f'CamelinaMAGIC{exp_num}.0-DailyTranspiration.csv')
df_transp = helpers.read_transpiration_csv(transp_file, RANDOMIZATION_FILE, exp_num)
df_transp = df_transp[df_transp['Treatment'] == 'DR_100']  # Filter for drought only

# Extract input features for this sample (Days 15-25)
INPUT_WINDOW = list(range(15, 26))
RECOVERY_WINDOW = list(range(36, 41))

sample_data = df_transp[df_transp['Sample'] == sample].sort_values('Days')
input_slice = sample_data[sample_data['Days'].isin(INPUT_WINDOW)]

if len(input_slice) < len(INPUT_WINDOW):
    print(f"❌ Error: Sample {sample} does not have complete input data (Days 15-25).")
else:
    # Get features
    X_sample = input_slice['Transpiration'].values.reshape(1, -1)
    
    # Load the trained model and scaler
    model_filename = f"LR_model_Cam_E{exp_num}_gen_{genotype.replace(' ', '_')}.pkl"
    scaler_filename = f"scaler_Cam_E{exp_num}_gen_{genotype.replace(' ', '_')}.pkl"
    model_path = os.path.join(MODELS_DIR, model_filename)
    scaler_path = os.path.join(MODELS_DIR, scaler_filename)
    
    if not os.path.exists(model_path) or not os.path.exists(scaler_path):
        print(f"❌ Error: Model or scaler not found.")
        print(f"   Expected: {model_path}")
        print(f"   Expected: {scaler_path}")
    else:
        # Load model and scaler
        with open(model_path, 'rb') as f:
            model = pickle.load(f)
        with open(scaler_path, 'rb') as f:
            scaler = pickle.load(f)
        
        # Normalize features using the loaded scaler
        X_sample_scaled = scaler.transform(X_sample)
        
        # Make prediction
        y_pred = model.predict(X_sample_scaled)[0]
        
        # Calculate actual slope from recovery window (Days 36-40)
        recovery_slice = sample_data[sample_data['Days'].isin(RECOVERY_WINDOW)]
        
        if len(recovery_slice) < 2:
            print(f"❌ Error: Sample {sample} does not have complete recovery data (Days 36-40).")
        else:
            from scipy.stats import linregress
            y_actual = float(linregress(recovery_slice['Days'], recovery_slice['Transpiration']).slope)
            
            # Calculate error metrics
            error = y_actual - y_pred
            abs_error = abs(error)
            percent_error = (abs_error / abs(y_actual) * 100) if y_actual != 0 else float('inf')
            
            # Display results
            print("\n" + "="*70)
            print(f"✓ INFERENCE RESULTS")
            print("="*70)
            print(f"Experiment:        {exp_num}")
            print(f"Genotype:          {genotype}")
            print(f"Sample:            {sample}")
            print(f"Input Period:      Days 15-25 (11 timesteps)")
            print(f"Target Period:     Days 36-40 (Recovery)")
            print("-"*70)
            print(f"Predicted Slope:   {y_pred:.6f} g/day")
            print(f"Actual Slope:      {y_actual:.6f} g/day")
            print("-"*70)
            print(f"Absolute Error:    {abs_error:.6f} g/day")
            print(f"Relative Error:    {percent_error:.2f}%")
            print(f"Error Direction:   {'Overestimated' if y_pred > y_actual else 'Underestimated'}")
            print("="*70 + "\n")
